# Tutorial 6: MOSTA FAST26 multi-section analysis

This tutorial shows how to prepare the FAST26 subset of the Mouse Organogenesis Spatiotemporal Transcriptomic Atlas (MOSTA) and fit a stage-aware multi-section GraphPCA-Turbo model. FAST26 contains 26 sagittal Stereo-seq sections from embryonic stages E12.5–E16.5. The full input is too large to distribute with the documentation, so the published page executes a small API smoke test and provides an explicit local path for the complete atlas.

Download the section-level AnnData files from the [MOSTA portal](https://db.cngb.org/stomics/mosta/download/). This tutorial expects raw counts in `layers["count"]`, two-dimensional coordinates in `obsm["spatial"]`, and an annotation column only for optional post hoc evaluation.

## Goal

By the end of the tutorial, you will be able to:

1. organize the 26 FAST26 files and their known developmental-stage labels;
2. select one shared stage-balanced gene panel;
3. normalize and scale each section while keeping only one section in working memory;
4. construct a within-section spatial graph and write a reusable disk-backed section store; and
5. run stage-aware GraphPCA-Turbo and load a section embedding on demand.

## Setup

Install GraphPCA-Turbo 2.2 or later in the analysis environment:

```bash
python -m pip install -U st-graphpca
```

The full run also requires `anndata`, `scanpy`, `numpy`, `pandas`, and `scipy`. The public page leaves `RUN_FULL_ATLAS` disabled. Set it to `True` only after assigning local paths and confirming that each FAST26 file contains the expected raw-count layer and spatial coordinates.

In [ ]:
from io import StringIO
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial import cKDTree

RUN_FULL_ATLAS = False
RUN_TINY_SMOKE_TEST = False  # Set True to test the installed v2.2 API on synthetic sections.
RAW_DIR = Path("/path/to/MOSTA_FAST26_h5ad")
STORE_DIR = Path("results/MOSTA_FAST26_section_store")
FIT_DIR = Path("results/MOSTA_FAST26_stage_grouped_fit")

# Ordered FAST26 manifest. Stage is a known sample-level label, not a fitted pseudotime.
_manifest = """order,file,stage
1,E12.5_E1S1.MOSTA.h5ad,E12.5
2,E12.5_E1S2.MOSTA.h5ad,E12.5
3,E12.5_E1S3.MOSTA.h5ad,E12.5
4,E12.5_E1S4.MOSTA.h5ad,E12.5
5,E12.5_E1S5.MOSTA.h5ad,E12.5
6,E12.5_E2S1.MOSTA.h5ad,E12.5
7,E13.5_E1S1.MOSTA.h5ad,E13.5
8,E13.5_E1S2.MOSTA.h5ad,E13.5
9,E13.5_E1S3.MOSTA.h5ad,E13.5
10,E13.5_E1S4.MOSTA.h5ad,E13.5
11,E14.5_E1S1.MOSTA.h5ad,E14.5
12,E14.5_E1S2.MOSTA.h5ad,E14.5
13,E14.5_E1S3.MOSTA.h5ad,E14.5
14,E14.5_E1S4.MOSTA.h5ad,E14.5
15,E14.5_E1S5.MOSTA.h5ad,E14.5
16,E14.5_E2S1.MOSTA.h5ad,E14.5
17,E14.5_E2S2.MOSTA.h5ad,E14.5
18,E15.5_E1S2.MOSTA.h5ad,E15.5
19,E15.5_E1S3.MOSTA.h5ad,E15.5
20,E15.5_E1S4.MOSTA.h5ad,E15.5
21,E15.5_E2S1.MOSTA.h5ad,E15.5
22,E15.5_E1S1.MOSTA.h5ad,E15.5
23,E16.5_E1S1.MOSTA.h5ad,E16.5
24,E16.5_E1S2.MOSTA.h5ad,E16.5
25,E16.5_E1S3.MOSTA.h5ad,E16.5
26,E16.5_E1S4.MOSTA.h5ad,E16.5"""
manifest = pd.read_csv(StringIO(_manifest))
manifest.head()

## 1. Validate the local input contract

Use one H5AD file per section. Gene names must be unique, and the selected genes must be present in every section. The code below opens files in backed mode, so validation itself does not materialize the atlas. Labels are not used to fit the representation; they are optional downstream metadata.

In [ ]:
required_stages = ["E12.5", "E13.5", "E14.5", "E15.5", "E16.5"]
assert len(manifest) == 26
assert manifest["file"].is_unique
assert set(manifest["stage"]) == set(required_stages)
print(manifest.groupby("stage").size().rename("n_sections"))

if RUN_FULL_ATLAS:
    import anndata as ad
    for row in manifest.itertuples(index=False):
        path = RAW_DIR / row.file
        if not path.exists():
            raise FileNotFoundError(path)
        section = ad.read_h5ad(path, backed="r")
        assert "count" in section.layers, f"{path}: missing layers['count']"
        assert "spatial" in section.obsm, f"{path}: missing obsm['spatial']"
        assert section.var_names.is_unique, f"{path}: duplicated genes"
        section.file.close()
    print("All FAST26 section contracts passed.")
else:
    print("Full-data validation skipped; assign RAW_DIR and set RUN_FULL_ATLAS=True.")

## 2. Preprocess one common gene panel

For the FAST26 analysis, start from raw counts, normalize each location to a total of 10,000, and apply `log1p`. Select a common panel from genes present in all 26 sections. To prevent stages with more sections from dominating the panel, score genes on a spatially balanced sample of up to 20,000 locations per section, retain a 1,500-gene recurrence-weighted consensus, then add 100 high-dispersion genes from each of the five stages. This yields 2,000 shared genes.

This step is deliberately separate from the model fit. Persist `selected_genes.tsv` and reuse it for every comparison on the same cohort.

In [ ]:
def normalize_log1p(counts, target_sum=10_000.0):
    """Return positive-library rows after sparse library normalization and log1p."""
    counts = sparse.csr_matrix(counts, dtype=np.float32)
    library_size = np.asarray(counts.sum(axis=1)).ravel()
    keep = np.isfinite(library_size) & (library_size > 0)
    counts = counts[keep]
    scale = sparse.diags((target_sum / library_size[keep]).astype(np.float32))
    values = (scale @ counts).tocsr()
    values.data = np.log1p(values.data).astype(np.float32, copy=False)
    return values, keep

def spatially_balanced_sample(coords, n_keep=20_000, seed=666, n_bins=20):
    """Sample locations approximately proportionally across a 2D grid."""
    n = len(coords)
    if n <= n_keep:
        return np.arange(n)
    xy = np.asarray(coords)[:, :2]
    span = np.maximum(xy.max(axis=0) - xy.min(axis=0), 1e-12)
    bins = np.minimum(n_bins - 1, ((xy - xy.min(axis=0)) / span * n_bins).astype(int))
    tile = bins[:, 0] * n_bins + bins[:, 1]
    rng = np.random.default_rng(seed)
    priority = rng.random(n)
    chosen = []
    for value in np.unique(tile):
        index = np.flatnonzero(tile == value)
        quota = min(len(index), max(1, round(n_keep * len(index) / n)))
        chosen.append(index[np.argsort(priority[index])[:quota]])
    chosen = np.unique(np.concatenate(chosen))
    return np.sort(chosen[np.argsort(priority[chosen])[:n_keep]])

# In a full run: for each section, use spatially_balanced_sample(), normalize_log1p(),
# and a normalized-dispersion score. Rank the common genes in each section; retain
# 1,500 recurrence-weighted consensus genes plus 100 additional top genes per stage.
# Save the ordered 2,000-gene result as selected_genes.tsv before building the store.

## 3. Build a disk-backed section store

The following functions make a radius graph independently inside each section and yield one processed section at a time. The radius is 1.45 times the median nearest-neighbor distance for that section. Then compute one equal-stage-weighted within-section standard deviation per gene, apply it in a second streaming pass, and write the section store. No spatial edge is created between sections.

For a reduced pilot, replace the all-location index with `spatially_balanced_sample(coords, n_keep=10_000, ...)`; for the full FAST26 fit, retain all nonempty locations.

In [ ]:
def radius_graph(coords, radius_factor=1.45):
    coords = np.asarray(coords, dtype=float)[:, :2]
    tree = cKDTree(coords)
    distances, _ = tree.query(coords, k=2)
    radius = radius_factor * float(np.median(distances[:, 1]))
    pairs = tree.query_pairs(radius, output_type="ndarray")
    rows = np.concatenate([pairs[:, 0], pairs[:, 1]])
    cols = np.concatenate([pairs[:, 1], pairs[:, 0]])
    return sparse.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                             shape=(len(coords), len(coords)))

def read_lognorm_section(path, selected_genes):
    import anndata as ad
    section = ad.read_h5ad(path, backed="r")
    gene_index = pd.Index(section.var_names.astype(str)).get_indexer(selected_genes)
    if (gene_index < 0).any():
        raise ValueError(f"{path} does not contain every selected gene.")
    counts = section.layers["count"][:, gene_index]
    values, keep = normalize_log1p(counts)
    coords = np.asarray(section.obsm["spatial"])[keep, :2]
    section.file.close()
    return values, coords

def equal_stage_gene_scale(manifest, selected_genes):
    variances = []
    weights = []
    per_stage = manifest["stage"].value_counts()
    for row in manifest.itertuples(index=False):
        values, _ = read_lognorm_section(RAW_DIR / row.file, selected_genes)
        mean = np.asarray(values.mean(axis=0)).ravel()
        second = np.asarray(values.multiply(values).mean(axis=0)).ravel()
        variances.append(np.maximum(second - mean**2, 0.0))
        weights.append(1.0 / per_stage[row.stage])
    weights = np.asarray(weights) / np.sum(weights)
    return np.sqrt(np.maximum(np.average(variances, axis=0, weights=weights), 1e-8))

if RUN_FULL_ATLAS:
    from GraphPCA import create_hierarchical_section_store

    selected_genes = pd.read_csv("selected_genes.tsv", sep="\t")["gene"].astype(str).tolist()
    gene_scale = equal_stage_gene_scale(manifest, selected_genes)

    def full_section_generator():
        for row in manifest.itertuples(index=False):
            values, coords = read_lognorm_section(RAW_DIR / row.file, selected_genes)
            values = values @ sparse.diags((1.0 / gene_scale).astype(np.float32))
            yield Path(row.file).stem, values, radius_graph(coords)

    store = create_hierarchical_section_store(
        STORE_DIR, full_section_generator(), expression_storage="auto"
    )
    print(store)
else:
    print("Store creation skipped. The code is ready after RAW_DIR and selected_genes.tsv are supplied.")

## 4. Fit stage-aware GraphPCA-Turbo

`out_of_core_group_labels=manifest["stage"]` partially pools each section toward the loading centre for its known embryonic stage. It does not use developmental order, infer pseudotime, or add cross-section spatial edges. A common rotation is retained so that embeddings have comparable programme coordinates.

The stage-balanced `sample_weights` below prevent a stage represented by more sections from receiving more total objective weight. Choose `rhos`, the number of components, and the stopping budget on a pilot appropriate to the study; the values below are an example configuration, not a universal optimum.

In [ ]:
if RUN_FULL_ATLAS:
    from GraphPCA import Run_Hierarchical_Multi_GPCA

    stage_labels = manifest["stage"].astype(str).tolist()
    stage_counts = pd.Series(stage_labels).value_counts()
    stage_balanced_weights = np.array([1.0 / stage_counts[label] for label in stage_labels])
    stage_balanced_weights /= stage_balanced_weights.sum()

    Z_disk, W_stage, W_sections, info = Run_Hierarchical_Multi_GPCA(
        adatas=None,
        execution_mode="out_of_core",
        section_store=STORE_DIR,
        out_of_core_output_dir=FIT_DIR,
        out_of_core_group_labels=stage_labels,
        n_components=15,
        lambdas=0.5,
        rhos=0.15,  # Illustrative partial-pooling strength; tune on a pilot.
        sample_weights=stage_balanced_weights,
        pcg_tol=1e-5,
        pcg_max_iter=250,
        outer_tol=1e-4,
        max_iter=50,
        mode="accelerated",
        return_info=True,
    )
    print(info.converged, info.n_iter, W_stage.shape, len(Z_disk))
    Z_E12_5_S1 = Z_disk.load(0, mmap_mode="r")
    print(Z_E12_5_S1.shape)
else:
    print("Full fit skipped. This mode reads one prepared section at a time.")

## Checks

Before a production run, check that the store has 26 sections, 2,000 genes, a nonempty graph per section, and stage labels in the same order as the store manifest. After fitting, inspect `info.converged` and `info.n_iter`; a stopped run can be continued with `out_of_core_resume=True` and the same output directory.

The optional tiny smoke test below is intentionally synthetic. It validates the public stage-aware API without claiming a biological result.

In [ ]:
if RUN_TINY_SMOKE_TEST:
    from GraphPCA import create_hierarchical_section_store, Run_Hierarchical_Multi_GPCA
    rng = np.random.default_rng(7)
    toy_sections = []
    for name in ["E12.5_demo_1", "E12.5_demo_2", "E13.5_demo_1"]:
        x = sparse.csr_matrix(rng.poisson(1.5, size=(40, 12)).astype(np.float32))
        graph = sparse.diags([np.ones(39), np.ones(39)], [-1, 1], shape=(40, 40))
        toy_sections.append((name, x, graph))
    with tempfile.TemporaryDirectory() as temp_dir:
        store = create_hierarchical_section_store(Path(temp_dir) / "store", toy_sections)
        z_disk, w_stage, w_sections, info = Run_Hierarchical_Multi_GPCA(
            adatas=None, execution_mode="out_of_core", section_store=store,
            out_of_core_output_dir=Path(temp_dir) / "fit",
            out_of_core_group_labels=["E12.5", "E12.5", "E13.5"],
            n_components=3, lambdas=0.2, rhos=0.1, max_iter=2,
            pcg_tol=1e-5, pcg_max_iter=100, return_info=True,
        )
        assert len(z_disk) == 3 and w_stage.shape == (2, 12, 3)
        assert z_disk.load(0).shape == (40, 3) and np.isfinite(info.objective_values[-1])
    print("Tiny stage-aware API smoke test passed.")
else:
    print("Tiny API smoke test skipped.")

## Next steps

Use the disk-backed `Z_disk.load(i, mmap_mode="r")` accessor to perform section-level downstream analyses without loading all embeddings simultaneously. Keep the selected gene panel, graph definition, section order, stage labels, solver settings, and output directory together as a reproducibility record.

Disk-backed fitting is an optional memory-saving execution mode. It remains section-sequential and retains global loading state, so benchmark the chosen component count, graph, and OpenMP thread limit on the target machine. Do not interpret the known stage labels as a fitted developmental trajectory.